# tensor-wraps-ndarray composite — cx3: rearrange-flatten on a wrapped-ndarray Tensor (no copy)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 3 atoms together: `einops-rearrange`, `einops-rearrange-flatten`, `tensor-wraps-ndarray`
> Running the final beacon reports progress against all 3 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "tensor-wraps-ndarray"
DD_ATOM_IDS = ["einops-rearrange", "einops-rearrange-flatten", "tensor-wraps-ndarray"]
DD_SUBTOPICS = ["Einops: Rearrange", "Einops: Rearrange-as-flatten", "PyTorch: tensor from ndarray"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these three atoms compose

ARENA's custom autograd uses a thin `Tensor` wrapper around a numpy ndarray (the `array` attribute). The three atoms compose like so:

1. **`tensor-wraps-ndarray`** — `Tensor.__init__` stores the ndarray directly. Reshapes inside `Tensor` ops should reuse the underlying buffer (no copy), so the wrapped `Tensor` is just a structured view.
2. **`einops-rearrange`** — the general rearrange pattern syntax. Works on any array-like, including raw ndarrays.
3. **`einops-rearrange-flatten`** — the `'b c h w -> b (c h w)'` grouped-axis flatten — the same op as cx1, but applied to the inner ndarray of a wrapped `Tensor` and wrapped back up.

**The composition.** Build a `Tensor.flatten_bchw()` method that calls `rearrange` on `self.array` (an ndarray), wraps the result back into a new `Tensor`, and the two wrappers SHARE storage.

### Composite Exercise — rearrange-flatten on a wrapped-ndarray Tensor (no copy)

**Atoms exercised together**: `einops-rearrange`, `einops-rearrange-flatten`, `tensor-wraps-ndarray`

A minimal `Tensor` class is provided at module scope (see the stub). It wraps a single numpy ndarray as `self.array`.

Build `cx3_flatten_bchw(ten)` that takes a `Tensor` whose `.array` has shape `(B, C, H, W)` and returns a NEW `Tensor` whose `.array` has shape `(B, C*H*W)`.

Constraints:
- Must use `einops.rearrange` on the inner ndarray with the `'b c h w -> b (c h w)'` pattern.
- The output `Tensor`'s `.array` MUST share memory with the input `Tensor`'s `.array` (no copy) — verified via `np.shares_memory(...)`.
- Mutating the input ndarray must be visible through the output.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

class Tensor:
    """Minimal ARENA-style Tensor: wraps a single ndarray, no metadata."""
    def __init__(self, array):
        assert isinstance(array, np.ndarray), 'Tensor wraps an ndarray'
        self.array = array

    def __repr__(self):
        return f'Tensor(shape={self.array.shape}, dtype={self.array.dtype})'

def cx3_flatten_bchw(ten):
    """Flatten (B, C, H, W) -> (B, C*H*W) on a wrapped Tensor, no copy."""
    raise NotImplementedError()

def _test_cx3():
    # --- (a) shape correct ---
    arr = np.arange(2 * 3 * 4 * 5, dtype=np.float32).reshape(2, 3, 4, 5)
    ten = Tensor(arr)
    out = cx3_flatten_bchw(ten)
    assert isinstance(out, Tensor), f'must return a Tensor, got {type(out)}'
    assert out.array.shape == (2, 60), f'expected (2,60), got {out.array.shape}'

    # --- (b) values match (c h w) inner-loop-fastest order ---
    expected = arr.reshape(2, 60)
    assert np.array_equal(out.array, expected), 'flatten order differs from row-major reshape'

    # --- (c) NO COPY: output ndarray shares memory with input ndarray ---
    assert np.shares_memory(out.array, ten.array), (
        'output array must share memory with the wrapped input — '
        'did you accidentally call .copy() or do an out-of-place op?')

    # --- (d) mutation through the wrapper is visible in the flattened view ---
    ten.array[0, 0, 0, 0] = -999.0
    assert out.array[0, 0] == -999.0, (
        'mutating input.array[0,0,0,0] should be visible at output.array[0,0] '
        'since they share storage')

    # --- (e) different shape ---
    arr2 = np.random.randn(1, 2, 3, 3).astype(np.float32)
    ten2 = Tensor(arr2)
    out2 = cx3_flatten_bchw(ten2)
    assert out2.array.shape == (1, 18)
    assert np.allclose(out2.array, arr2.reshape(1, 18))
    assert np.shares_memory(out2.array, ten2.array)
    _dd_passed.add('cx3')

_test_cx3()

<details><summary>Show solution — cx3</summary>

```python
def cx3_flatten_bchw(ten):
    flat = rearrange(ten.array, 'b c h w -> b (c h w)')
    # einops on a contiguous ndarray returns a reshape view — same buffer.
    return Tensor(flat)
```

All three atoms compose into a four-line function: the `Tensor(...)` wrapper (tensor-wraps-ndarray) holds an ndarray; `rearrange(...)` (einops-rearrange) is the call; the `(c h w)` grouped-axis (einops-rearrange-flatten) is the pattern. Calling `.copy()` or swapping in `np.asarray(...).reshape(...).copy()` breaks the `shares_memory` check in test (c).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 3 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx3',
        'subtopics': ["Einops: Rearrange", "Einops: Rearrange-as-flatten", "PyTorch: tensor from ndarray"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()